# 10.7 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/07-responsible-ai/solutions.ipynb)

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


## Exercise 1

In [2]:
toy = pd.DataFrame({"group": list("CCCCCCCCDDDD"),
                    "flag":    [1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0],
                    "churned": [1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0]})
for name, g in toy.groupby("group"):
    churn = g["churned"] == 1
    print(f"{name}: flag rate {g['flag'].mean():.2f}, recall {(g['flag'][churn] == 1).mean():.2f}, "
          f"FPR {(g['flag'][~churn] == 1).mean():.2f}")

C: flag rate 0.50, recall 0.67, FPR 0.40
D: flag rate 0.25, recall 0.50, FPR 0.00


C: flag rate 0.50, recall 0.67, FPR 0.40. D: flag rate 0.25, recall 0.50, FPR 0.00. The demographic-parity gap (0.25) is larger than the equal-opportunity gap (0.17). Group D's churners are caught less often, and its loyal customers are never bothered: which matters more depends on whether a voucher is a benefit or a nuisance.

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.base import clone
cust = cm.build_customer_table(cm.load_sales())
train, test = train_test_split(cust, test_size=0.25, random_state=42, stratify=cust["churned"])
test = test.assign(p=bundle["pipeline"].predict_proba(test[cm.FEATURES])[:, 1],
                   group=np.where(test["country"] == "United Kingdom", "UK", "non-UK"))
print(test["group"].value_counts().to_string())

def group_metrics(df, threshold):
    """Flag rate, recall (true positive rate) and false positive rate per group."""
    rows = {}
    for name, g in df.groupby("group"):
        flag, churn = g["p"] >= threshold, g["churned"] == 1
        rows[name] = {"customers": len(g), "churners": int(churn.sum()), "flag rate": flag.mean(),
                      "recall": (flag & churn).sum() / churn.sum(),
                      "false positive rate": (flag & ~churn).sum() / (~churn).sum(),
                      "avg predicted": g["p"].mean(), "actual churn": churn.mean()}
    return pd.DataFrame(rows).T

group
UK        749
non-UK     81


## Exercise 2

In [4]:
sub = test[test["country"].isin(["Germany", "France"])].assign(group=lambda d: d["country"])
print(group_metrics(sub, 0.35).round(2).to_string())

         customers  churners  flag rate  recall  false positive rate  avg predicted  actual churn
France        17.0       6.0       0.65    0.83                 0.55           0.43          0.35
Germany       13.0       4.0       0.62    0.75                 0.56           0.40          0.31


Each country has only a handful of churners in the test set. Any difference in recall here is a few customers either way: nothing can be concluded. The right report says exactly that, and recommends pooling several months of data before judging.

## Exercise 3

In [5]:
p_cv = cross_val_predict(clone(bundle["pipeline"]), train[cm.FEATURES], train["churned"],
                         cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]
train_g = train.assign(p=p_cv, group=np.where(train["country"] == "United Kingdom", "UK", "non-UK"))
thresholds = {}
for name, g in train_g.groupby("group"):
    churn_p = np.sort(g.loc[g["churned"] == 1, "p"])
    thresholds[name] = float(churn_p[int(0.20 * len(churn_p))])     # 80% of churners score above it
print({k: round(v, 3) for k, v in thresholds.items()})
for name, g in test.groupby("group"):
    flag, churn = g["p"] >= thresholds[name], g["churned"] == 1
    print(f"{name}: recall {(flag & churn).sum() / churn.sum():.2f}, flag rate {flag.mean():.2f}")

{'UK': 0.404, 'non-UK': 0.41}
UK: recall 0.80, flag rate 0.60
non-UK: recall 0.77, flag rate 0.54


Group-specific thresholds can equalise recall on training data, but on 30 non-UK churners the test recall still swings by chance, and the flag rates move apart. Using a group attribute to set a decision is also a legal and ethical question in its own right (in many settings it is not allowed for protected characteristics). Here the measured gap is within noise, so the defensible choice is one threshold for everyone, plus monitoring.